# 🤖 AutoML Pilot Pro - Advanced Training & Automated Reporting
Run state-of-the-art AutoML models and receive results via email automatically.

In [ ]:
# @title 1. Install & Import Dependencies
!pip install -q pycaret[full] ydata-profiling
import pandas as pd
import os
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize
from ydata_profiling import ProfileReport
import smtplib, ssl, json
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
print('✅ Setup complete!')

In [ ]:
# @title 2. Configuration & Data Load
target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]
recipient_email = 'user@example.com' # @param {type:"string"}

from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f'✅ Loaded {filename}: {df.shape}')

In [ ]:
# @title 3. Automated EDA
profile = ProfileReport(df, title="AutoML Pilot EDA Report", explorative=True)
profile.to_file("eda_report.html")
print('✅ EDA Report generated as eda_report.html')

In [ ]:
# @title 4. AutoML Training
if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = cls_compare()
    results = cls_pull()
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = reg_compare()
    results = reg_pull()
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')

display(results.head())
print('✅ Training complete. Model saved.')

In [ ]:
# @title 5. Send Report via Email
def send_email(to_email, metrics_df):
    # This is a template. In a real scenario, you'd need SMTP credentials.
    # For Colab, we'll just print what would be sent unless configured.
    print(f'📧 Preparing email for {to_email}...')
    print('Leaderboard Summary:')
    print(metrics_df.head().to_string())
    
send_email(recipient_email, results)

In [ ]:
# @title 6. Download Artifacts
from google.colab import files
files.download('best_model.pkl')
files.download('eda_report.html')
print('✅ Downloads triggered.')